## Topic: LangSmith Introduction

### Agenda
- 1. Introduction of LangSmith

- 2. Why do we need LangSmith

- 3. The LangSmith UI: Dashboard Walkthrough

- 4. Run Trees and Span Hierarchy

- 5. Key Takeaways

### 1. Introduction of LangSmith

- Definition:
    - LangSmith is a unified platform for debugging, testing, evaluating, and monitoring LLM applications. Built by the LangChain team, it gives you full visibility into every step of your LLM pipeline — from the raw prompt sent to the model, to the exact tokens returned, to the latency and cost of each call.

- Key Rule of Thump:
    - LangSmith is a tools that give the observability functionality of LLM application.

### 2. Why do we need LangSmith

- The Problem without the LangSmith
    - 1. Latency (increase)
        - we can't saw why the latency is high in our application.
        - why are application is slow based on some specific queries. 
        - eg: Al job Application Assistant (Read the jod, search the user portfolio, mathching the relevant skills to the job and so on...)

    - 2. Cost (increase)
        - Why the cost are increase in our application we do't know.
        - debug is very difficult.
        - How many tokens are use
        - eg: Research Assistant 
            - summarize the 5 latest papers on user query
            - search academic sources for relevant papers
            - Read each paper and extract key points.
            - Generate a detailed summary and so on..
    
    - 3. Hallucination
        - eg: RAG-based application
        
        - The application can be give the misinformation.

        - A RAG-based application two way to provided misinformation or Hallucination
            - 1. Retrieval can't fetch the relevant information.
                - number of retrieve is too low (k=1).

            - 2. Generator (LLm responding)
                - low-quality llm 
                - prompt is not good. 
                    - prompt is linear
                    - prompt is not strongly empathize 
    

### 3. Why Observability Matters in LLM Applications

- Observability is the ability to understand a system's internal state by examining it's external outputs, like logs, metrics, and traces.

- It allows you to diagnose issue, understand performance, and improve reliability by analyzing data generated by the system.

- Essentially, it's about being able to answer "why" something is happening within a system, even if you didn't anticipate the problem.

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│           WHY OBSERVABILITY MATTERS                         │
│                                                             │
│  1. LLMs ARE NON-DETERMINISTIC                              │
│     The same prompt can produce different outputs each time.│
│     You need traces to understand WHY a specific output     │
│     was generated on a specific run.                        │
│                                                             │
│  2. PROMPTS ARE THE NEW CODE                                │
│     A single word change in a prompt can break your app.    │
│     You need versioning, diffing, and rollback — just       │
│     like you do with source code.                           │
│                                                             │
│  3. CHAINS HAVE MANY FAILURE POINTS                         │
│     Retriever returns wrong docs? Reranker drops the right  │
│     one? LLM hallucinates? Tool returns garbage?            │
│     Without traces, you can't tell WHICH step failed.       │
│                                                             │
│  4. COSTS ARE HIDDEN AND UNPREDICTABLE                      │
│     A single user query might trigger 1, 5, or 20 LLM calls │
│     depending on the agent's behavior. You need visibility  │
│     into per-query cost to avoid surprise bills.            │
│                                                             │
│  5. LATENCY IS A USER EXPERIENCE ISSUE                      │
│     Users abandon apps that take > 3 seconds to respond.    │
│     You need to know which step is the bottleneck.          │
│                                                             │
│  6. QUALITY DEGRADES SILENTLY                               │
│     Your app worked great last month. Is it still good?     │
│     Without systematic evaluation, quality drift goes       │
│     unnoticed until users complain.                         │
│                                                             │
│  7. DEBUGGING LLMs IS FUNDAMENTALLY DIFFERENT               │
│     Traditional debugging: set breakpoints, inspect vars.   │
│     LLM debugging: inspect prompts, completions, context,   │
│     retrieved documents, tool inputs/outputs.               │
│     You need a purpose-built tool.                          │
└─────────────────────────────────────────────────────────────┘

"""

### 4. Run Trees and Span Hierarchy

- What does LanSmith Trace?
    - 1. Input and output

    - 2. All the intermediate steps

    - 3. Latency

    - 4. Token usage

    - 5. Cost

    - 6. Error

    - 7. Tags

    - 8. Metadata

    - 9. Feedback

- Core Concepts
- workflow: user --> prompt --> LLM ---> Parser
    - 1. Project
        - The entire workflow is called project.

    - 2. Trace
        - One simple execution is called Trace.

    - 3. Run
        - The Workflow is build multiple runnable components, each runnable component are called Run.
        - for above workflow:
            - Run 1: Prompt
            - Run 2: LLM
            - Run 3: Parser

In [ ]:
""" 
- Every trace is a tree of runs. The root run represents the full invocation, and child runs represent individual steps.

1. Run Tree Structure:

  Root Run (Chain / Agent invocation)
  ├── Run: Retriever (fetch documents)
  │   └── Run: Embeddings API call
  ├── Run: Reranker (score documents)
  ├── Run: LLM (generate answer)
  │   ├── Input:  "You are a helpful... Context: ... Question: ..."
  │   ├── Output: "Based on the documents, the answer is..."
  │   ├── Tokens: 1,234 input + 567 output
  │   ├── Latency: 1.2s
  │   └── Cost: $0.003
  └── Run: Output Parser (format response)

  Each run contains:
  ┌───────────────────────────────────────────┐
  │  Run Object                               │
  │  ├── name: "ChatOpenAI"                   │
  │  ├── run_type: "llm" | "chain" | "tool"   │
  │  ├── inputs: {...}                        │
  │  ├── outputs: {...}                       │
  │  ├── start_time: 2024-01-15T10:30:00Z     │
  │  ├── end_time: 2024-01-15T10:30:01Z       │
  │  ├── error: null | "Error message"        │
  │  ├── latency: 1.2s                        │
  │  ├── tokens: {prompt: 1234, completion: 567}│
  │  ├── cost: $0.003                         │
  │  ├── tags: ["production", "rag"]          │
  │  ├── metadata: {"user_id": "123"}         │
  │  └── child_runs: [...]                    │
  └───────────────────────────────────────────┘

  
===================================================================

  - 2. Project: example-simple-chain
  ┌─────────────────────────────────────────────────────┐
  │  Trace: RunnableSequence                            │
  │  Duration: 1.4s | Tokens: 89 in, 156 out            │
  │                                                     │
  │  ├── ChatPromptTemplate (0ms)                       │
  │  │   Input:  {role: "python expert", ...}           │
  │  │   Output: [SystemMessage, HumanMessage]          │
  │  │                                                  │
  │  ├── ChatOpenAI (1.3s) ← Most of the time!          │
  │  │   Input:  [SystemMessage("You are a python..."), │
  │  │            HumanMessage("What is a decorator?")] │
  │  │   Output: AIMessage("A decorator is a...")       │
  │  │   Model:  gpt-4o-mini                            │
  │  │   Temp:   0                                      │
  │  │   Tokens: 89 prompt + 156 completion             │
  │  │   Cost:   $0.0004                                │
  │  │                                                  │
  │  └── StrOutputParser (0ms)                          │
  │      Input:  AIMessage("A decorator is a...")       │
  │      Output: "A decorator is a..."                  │
  └─────────────────────────────────────────────────────┘

  You can click into ANY step to see the exact
  inputs and outputs. No more guessing!


===================================================================
- 3. What You'll See in LangSmith (Agent Trace):

  Project: example-langgraph-agent
  ┌─────────────────────────────────────────────────────────┐
  │  Trace: LangGraph (StateGraph)                          │
  │  Duration: 3.8s | Total Tokens: 1,234 | Cost: $0.005   │
  │                                                         │
  │  ├── Step 1: agent (LLM Call #1) — 1.2s                │
  │  │   Input:  "What's the weather in New York..."        │
  │  │   Output: tool_calls: [search_weather("New York"),   │
  │  │                        search_population("New York")]│
  │  │   Note: LLM decided to call TWO tools!               │
  │  │                                                      │
  │  ├── Step 2: tools (Tool Execution) — 0.1s              │
  │  │   ├── search_weather("New York")                     │
  │  │   │   Output: "72°F and sunny"                       │
  │  │   └── search_population("New York")                  │
  │  │       Output: "8.3 million"                          │
  │  │                                                      │
  │  ├── Step 3: agent (LLM Call #2) — 1.5s                │
  │  │   Input:  [original question + tool results]         │
  │  │   Output: "The weather in New York is 72°F..."       │
  │  │   Note: LLM synthesized the final answer!            │
  │  │                                                      │
  │  └── END                                                │
  └─────────────────────────────────────────────────────────┘

  KEY INSIGHT:
  Without LangSmith, you'd only see the final answer.
  With LangSmith, you see the ENTIRE reasoning process:
  which tools were called, what they returned, and how
  the LLM combined the results.


"""

- Install:
    - 1. pip install langsmith
    - or pip install -r requirements.txt

- Create an account on LangSmith

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│           ENVIRONMENT VARIABLES REFERENCE                   │
│                                                             │
│  LANGSMITH_API_KEY      (Required) Your API key             │
│  LANGSMITH_TRACING      (Required) "true" to enable tracing │
│  LANGSMITH_PROJECT      (Optional) Project name for traces  │
│  LANGSMITH_ENDPOINT     (Optional) API URL (for self-hosted)│
│  LANGCHAIN_TRACING_V2   (Legacy)   Use LANGSMITH_TRACING    │
│  LANGCHAIN_API_KEY      (Legacy)   Use LANGSMITH_API_KEY    │
│  LANGCHAIN_PROJECT      (Legacy)   Use LANGSMITH_PROJECT    │
│                                                             │
│  NOTE: The LANGCHAIN_* variables still work for backward    │
│  compatibility, but LANGSMITH_* is the new standard.        │
└─────────────────────────────────────────────────────────────┘
"""

### 5. Key Takeaways

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│           LANGSMITH UI — MAIN SECTIONS                      │
│                                                             │
│  📊 PROJECTS (Home)                                         │
│  ──────────────────                                         │
│  List of all your projects. Each project groups traces      │
│  from a specific app or environment.                        │
│  • my-chatbot-prod    (1,234 traces today)                  │
│  • my-chatbot-staging (56 traces today)                     │
│  • rag-pipeline-v2    (890 traces today)                    │
│                                                             │
│  🔍 TRACES (The Core View)                                 │
│  ──────────────────────                                     │
│  A searchable, filterable list of all runs.                 │
│  • Click any trace to expand the full run tree              │
│  • See inputs, outputs, latency, tokens, errors             │
│  • Filter by: time range, status, tags, metadata            │
│  • Sort by: latency, tokens, cost, time                     │
│                                                             │
│  📈 MONITORING (Charts & Trends)                            │
│  ──────────────────────────────                             │
│  Time-series charts for:                                    │
│  • Latency (p50, p95, p99)                                  │
│  • Error rate                                               │
│  • Token usage                                              │
│  • Cost per day                                             │
│  • Feedback scores                                          │
│                                                             │
│  📋 DATASETS (Test Data)                                    │
│  ──────────────────────                                     │
│  Manage your evaluation datasets.                           │
│  • Create from scratch or from production traces            │
│  • Add examples with inputs and expected outputs            │
│  • Version your datasets                                    │
│                                                             │
│  🧪 EXPERIMENTS (Evaluation Results)                        │
│  ──────────────────────────────                             │
│  Compare evaluation runs side by side.                      │
│  • "gpt-4o vs gpt-4o-mini on QA dataset"                    │
│  • "prompt-v1 vs prompt-v2 on summarization"                │
│  • See score distributions, pass/fail rates                 │
│                                                             │
│  💬 PROMPT HUB (Prompt Management)                          │
│  ──────────────────────────────                             │
│  Version and share prompts.                                 │
│  • Commit prompt changes like code                          │
│  • Roll back to previous versions                           │
│  • Share across team members                                │
│                                                             │
│  ⚙️ SETTINGS                                                │
│  ──────────────                                             │
│  API keys, team members, billing, data retention.           │
└─────────────────────────────────────────────────────────────┘

"""